In [ ]:
import inspect
import json
import math
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

output_dir = Path('/kaggle/working/results')
output_dir.mkdir(parents=True, exist_ok=True)
dataset_dir = Path('/kaggle/input/datasets/priyansh2006/soap-pinn')
schedule_path = dataset_dir / 'poisson_seed0_K10_schedule.pt'
paper_source_path = dataset_dir / 'paper_soap.py'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
optimizer = 'paper_soap_style'
experiment_type = 'gradient extrapolation reliability diagnostic K10'
initial_factor_mode = 'first_gradient'
update_order = 'after_update'
steps = 500
eval_every = 1
seed = 0
K = 10
n_pde = 1000
n_boundary_each = 100
lr = 1e-3
beta2 = 0.95
damping = 1e-3
precondition_frequency = 1
max_precond_dim = 10000
prediction_mus = [0.0, 0.25, 0.5, 0.85, 1.0]
schedule_path = output_dir / 'poisson_seed0_K10_schedule.pt'
output_name = 'stage2_paper_soap_style_gradient_prediction_seed0_K10_500.json'

print({
    'optimizer': optimizer,
    'experiment_type': experiment_type,
    'initial_factor_mode': initial_factor_mode,
    'update_order': update_order,
    'steps': steps,
    'eval_every': eval_every,
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'beta2': beta2,
    'damping': damping,
    'precondition_frequency': precondition_frequency,
    'max_precond_dim': max_precond_dim,
    'prediction_mus': prediction_mus,
    'schedule': str(schedule_path),
    'paper_source': str(paper_source_path),
    'output': str(output_dir / output_name),
})

In [ ]:
if not paper_source_path.exists():
    raise FileNotFoundError(f'paper_soap.py not found at {paper_source_path}')
sys.path.insert(0, str(paper_source_path.parent))
from paper_soap import PaperSOAPStyle

paper_source = inspect.getsourcefile(PaperSOAPStyle)
print('Paper SOAP source:', paper_source)
if Path(paper_source).resolve() != paper_source_path.resolve():
    raise RuntimeError(f'Unexpected Paper SOAP source: {paper_source}')

In [ ]:
class PINN(nn.Module):
    def __init__(self, width=128, hidden_layers=4):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers.extend([nn.Linear(width, width), nn.Tanh()])
        layers.append(nn.Linear(width, 1))
        self.net = nn.Sequential(*layers)
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, xy):
        return self.net(xy)


def set_seed(value):
    torch.manual_seed(value)
    np.random.seed(value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(value)

In [ ]:
def poisson_residual(model, xy):
    prediction = model(xy)
    first = torch.autograd.grad(
        prediction, xy, torch.ones_like(prediction), create_graph=True, retain_graph=True
    )[0]
    u_xx = torch.autograd.grad(
        first[:, 0:1], xy, torch.ones_like(first[:, 0:1]), create_graph=True, retain_graph=True
    )[0][:, 0:1]
    u_yy = torch.autograd.grad(
        first[:, 1:2], xy, torch.ones_like(first[:, 1:2]), create_graph=True, retain_graph=True
    )[0][:, 1:2]
    forcing = 2.0 * math.pi ** 2 * torch.sin(math.pi * xy[:, 0:1]) * torch.sin(math.pi * xy[:, 1:2])
    return -(u_xx + u_yy) - forcing


def relative_l2(model):
    grid = torch.linspace(0.0, 1.0, 81, device=device)
    xx, yy = torch.meshgrid(grid, grid, indexing='ij')
    xy = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    with torch.no_grad():
        prediction = model(xy).reshape(-1)
    reference = torch.sin(math.pi * xy[:, 0]) * torch.sin(math.pi * xy[:, 1])
    return float(torch.linalg.vector_norm(prediction - reference) / (torch.linalg.vector_norm(reference) + 1e-12))


def flat_grad(model):
    gradients = [parameter.grad.detach().reshape(-1) for parameter in model.parameters() if parameter.grad is not None]
    if not gradients:
        raise RuntimeError('No gradients available')
    return torch.cat(gradients)


def safe_min_eigenvalue(matrix):
    symmetric = ((matrix + matrix.T) * 0.5).detach().to(dtype=torch.float64, device='cpu')
    if not torch.isfinite(symmetric).all():
        return float('nan'), 'non-finite factor'
    try:
        value = torch.linalg.eigvalsh(symmetric).min().item()
        return value, None
    except RuntimeError as error:
        return float('nan'), str(error)

In [ ]:
def sample_boundary(n_each):
    values = torch.rand(4 * n_each, device=device)
    boundary = torch.empty(4 * n_each, 2, device=device)
    boundary[:n_each] = torch.stack([torch.zeros(n_each, device=device), values[:n_each]], dim=1)
    boundary[n_each:2 * n_each] = torch.stack([torch.ones(n_each, device=device), values[n_each:2 * n_each]], dim=1)
    boundary[2 * n_each:3 * n_each] = torch.stack([values[2 * n_each:3 * n_each], torch.zeros(n_each, device=device)], dim=1)
    boundary[3 * n_each:] = torch.stack([values[3 * n_each:], torch.ones(n_each, device=device)], dim=1)
    return boundary


def make_reuse_schedule(schedule_length):
    schedule = []
    current_pde = None
    current_boundary = None
    for step in range(1, schedule_length + 1):
        if step == 1 or (step - 1) % K == 0:
            current_pde = torch.rand(n_pde, 2, device=device)
            current_boundary = sample_boundary(n_boundary_each)
        schedule.append({
            'pde': current_pde.detach().clone(),
            'bc': current_boundary.detach().clone(),
        })
    return schedule


if schedule_path.exists():
    schedule = torch.load(schedule_path, map_location=device, weights_only=True)
    print(f'Loaded frozen K={K} schedule:', schedule_path)
else:
    set_seed(seed)
    schedule = make_reuse_schedule(2000)
    torch.save(schedule, schedule_path)
    print(f'Created frozen K={K} schedule:', schedule_path)

print('Schedule length:', len(schedule))
if len(schedule) != 2000:
    raise ValueError(f'Expected schedule length 2000, found {len(schedule)}')

for block_start in range(0, 2000, K):
    block_end = min(block_start + K, 2000)
    reference = schedule[block_start]
    for index in range(block_start + 1, block_end):
        if not (
            torch.equal(reference['pde'], schedule[index]['pde'])
            and torch.equal(reference['bc'], schedule[index]['bc'])
        ):
            raise ValueError(f'Schedule batch changed inside K={K} reuse block at step {index + 1}')
print(f'Validated batch reuse blocks of K={K}.')

In [ ]:
set_seed(seed)
model = PINN().to(device)
optimizer_instance = PaperSOAPStyle(
    model.parameters(),
    lr=lr,
    beta2=beta2,
    damping=damping,
    precondition_frequency=precondition_frequency,
    max_precond_dim=max_precond_dim,
    initial_factor_mode=initial_factor_mode,
    update_order=update_order,
)
print('Paper-SOAP-style optimizer initialized.')

In [ ]:
history = []
previous_gradient = None
previous_previous_gradient = None
start_time = time.perf_counter()
model.train()

for step in range(1, steps + 1):
    current_pde = schedule[step - 1]['pde'].detach().clone().requires_grad_(True)
    current_boundary = schedule[step - 1]['bc']
    optimizer_instance.zero_grad(set_to_none=True)

    loss_pde = torch.mean(poisson_residual(model, current_pde).square())
    loss_boundary = torch.mean(model(current_boundary).square())
    loss = loss_pde + loss_boundary
    if not torch.isfinite(loss):
        print(f'NON-FINITE LOSS AT STEP {step}')
        break
    loss.backward()

    gradient = flat_grad(model)
    if not torch.isfinite(gradient).all():
        print(f'NON-FINITE GRADIENT BEFORE STEP {step}')
        break
    gradient_norm = torch.linalg.vector_norm(gradient).item()
    grad_diff_norm = (
        torch.linalg.vector_norm(gradient - previous_gradient).item()
        if previous_gradient is not None else float('nan')
    )
    grad_change_ratio = (
        grad_diff_norm / (torch.linalg.vector_norm(gradient).item() + 1e-12)
        if previous_gradient is not None else float('nan')
    )
    grad_cosine = (
        torch.dot(previous_gradient, gradient).item()
        / (torch.linalg.vector_norm(previous_gradient).item() * gradient_norm + 1e-12)
        if previous_gradient is not None else float('nan')
    )

    prediction_metrics = {}
    if previous_gradient is not None and previous_previous_gradient is not None:
        delta = previous_gradient - previous_previous_gradient
        current_norm = gradient_norm
        for mu in prediction_mus:
            predicted_gradient = previous_gradient + mu * delta
            predicted_norm = torch.linalg.vector_norm(predicted_gradient).item()
            prediction_cosine = (
                torch.dot(predicted_gradient, gradient).item()
                / (predicted_norm * current_norm + 1e-12)
            )
            prediction_error = torch.linalg.vector_norm(predicted_gradient - gradient).item() / (current_norm + 1e-12)
            key = f'mu_{str(mu).replace(".", "").replace("0", "0", 1)}'
            if mu == 0.0:
                key = 'mu_0'
            elif mu == 0.25:
                key = 'mu_025'
            elif mu == 0.5:
                key = 'mu_050'
            elif mu == 0.85:
                key = 'mu_085'
            elif mu == 1.0:
                key = 'mu_100'
            prediction_metrics[key] = {
                'cosine': prediction_cosine,
                'relative_error': prediction_error,
            }

    previous_previous_gradient = previous_gradient
    previous_gradient = gradient.clone()

    parameters_before = torch.cat([
        parameter.detach().reshape(-1)
        for parameter in model.parameters()
    ])
    optimizer_instance.step()
    parameters_after = torch.cat([
        parameter.detach().reshape(-1)
        for parameter in model.parameters()
    ])
    update_norm = torch.linalg.vector_norm(parameters_after - parameters_before).item()

    if not torch.isfinite(parameters_after).all():
        print(f'NON-FINITE PARAMETERS AFTER STEP {step}')
        print('update_norm =', update_norm)
        break

    precond_gain_by_parameter = {}
    staleness_by_parameter = {}
    for name, parameter in model.named_parameters():
        state = optimizer_instance.state.get(parameter, {})
        if state.get('mode') == 'structured':
            precond_gain_by_parameter[name] = state.get('precond_gain')
            staleness_by_parameter[name] = state.get('staleness_error')

    finite_gains = [value for value in precond_gain_by_parameter.values() if value is not None and math.isfinite(value)]
    finite_staleness = [value for value in staleness_by_parameter.values() if value is not None and math.isfinite(value)]
    max_precond_gain = max(finite_gains) if finite_gains else float('nan')
    max_staleness_error = max(finite_staleness) if finite_staleness else float('nan')

    row = {
        'step': step,
        'loss': loss.item(),
        'loss_pde': loss_pde.item(),
        'loss_bc': loss_boundary.item(),
        'rel_l2': relative_l2(model),
        'grad_norm': gradient_norm,
        'grad_diff_norm': grad_diff_norm,
        'grad_change_ratio': grad_change_ratio,
        'grad_cosine': grad_cosine,
        'param_update_norm': update_norm,
        'precond_gain': precond_gain_by_parameter,
        'staleness_error': staleness_by_parameter,
        'max_precond_gain': max_precond_gain,
        'max_staleness_error': max_staleness_error,
        'prediction': prediction_metrics,
        'effective_update': True,
        'elapsed_sec': time.perf_counter() - start_time,
    }
    history.append(row)
    print(
        f'step={step:3d} loss={loss.item():.6e} '
        f'grad_norm={gradient_norm:.6e} update_norm={update_norm:.6e} '
        f'grad_cosine={grad_cosine:.6e} predictions={len(prediction_metrics)}'
    )

In [ ]:
result = {
    'stage': 2,
    'optimizer': optimizer,
    'implementation': 'paper SOAP-style reconstruction',
    'experiment_type': experiment_type,
    'initial_factor_mode': initial_factor_mode,
    'update_order': update_order,
    'paper_source': paper_source,
    'seed': seed,
    'K': K,
    'n_pde': n_pde,
    'n_boundary_each': n_boundary_each,
    'lr': lr,
    'beta2': beta2,
    'damping': damping,
    'precondition_frequency': precondition_frequency,
    'max_precond_dim': max_precond_dim,
    'prediction_mus': prediction_mus,
    'steps': steps,
    'eval_every': eval_every,
    'schedule_file': schedule_path.name,
    'device': str(device),
    'torch_version': torch.__version__,
    'history': history,
}
result_path = output_dir / output_name
result_path.write_text(json.dumps(result, indent=2))
print(f'Saved: {result_path}')

In [ ]:
print(f'Result file: {result_path}')
print(f'Paper SOAP source: {result["paper_source"]}')
print(f'Completed diagnostic steps: {len(history)}')
if history:
    print(f'Final metrics: {history[-1]}')